In [1]:
import pandas as pd

no_call = pd.read_csv("ecoh_no_call_by_cluster.tsv", sep="\t")

In [2]:
no_call = no_call[no_call.reason == "no_ecoh_hits"]

In [3]:
serotypes = pd.read_csv("oag_locus_serotype.csv")

In [4]:
set(serotypes.cluster) & set(no_call.cluster)

{32, 73, 137, 160}

In [5]:
unique_clusters = list(set(no_call.cluster))
unique_clusters.remove(32)
unique_clusters.remove(73)
unique_clusters.remove(137)
unique_clusters.remove(160)

In [9]:
from Bio import SeqIO
genbank_files = {}
i = 0
capsule_loci = []
# Iterate through records in a file
for record in SeqIO.parse("O-antigen_db.transposase_clean.min3orfs_rc.gbff", "genbank"):
    likely_capsule = False
    for feature in record.features:
        if feature.type == "CDS":
            if "wzi" in feature.qualifiers["product"][0].lower():
                likely_capsule = True
                i += 1
            elif "wza" in feature.qualifiers["product"][0].lower():
                likely_capsule = True
            elif "serotype k" in feature.qualifiers["product"][0].lower():
                likely_capsule = True
                i +=1

    if not likely_capsule:
        genbank_files[record.id] = record
    else:
        capsule_loci.append(record.id)


In [11]:
locus_capsule = no_call[no_call.locus.isin(capsule_loci)]

In [7]:
novel_otypes = []
for cluster in unique_clusters:
    locus = no_call[no_call.cluster == cluster]
    locus_name = locus.iloc[0].locus
    if genbank_files.get(locus_name) is not None:
        record = genbank_files[locus_name]
        for feature in record.features:
            if feature.type == "source":
                feature.qualifiers["serotype"] = [f"Ois{cluster}"]

        novel_otypes.append(record)
        SeqIO.write(record, f"Novel_Otypes_rep/{record.id}.gbk", "genbank")
SeqIO.write(novel_otypes, "Novel_Otypes_rep/combined_novel_otypes.gbff", "genbank")

58

In [17]:
no_call.cluster.value_counts()

cluster
31     14
209    13
16     13
131    13
178    12
       ..
242     1
53      1
155     1
169     1
84      1
Name: count, Length: 68, dtype: int64